# MASLD-LLM · C3 evaluations — v4 (hardened)

Same as v3 plus: adapter-dir FUSE probe + torchao re-check inside the
0.6B cell (hosted runtimes restore preinstalled torchao 0.10, and Colab's
Drive FUSE can leave checkpoint subdirs unmaterialized in fresh runtimes).

Evaluates 1.7B-v5 (and optionally 0.6B-v5) on ALL C3 corpora in Drive:
- MB-85xx progression
- MB-87xx liver->plasma (freeze-11 masked-protein 8705)
- MB-88xx PHH FFA perturbation (freeze-12)

Add a corpus = one line in CORPORA. Valid __meta__ headers at source.

In [1]:
# Cell 1: Mount Drive + locate the masld-llm workspace
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MASLD = '/content/drive/MyDrive/masld-llm'
assert os.path.isdir(DRIVE_MASLD), f'not found: {DRIVE_MASLD} — check the Drive account'
PROMPTS = os.path.join(DRIVE_MASLD, 'prompts')
OUT_DIR = os.path.join(DRIVE_MASLD, 'results', 'frontier_runs')
os.makedirs(OUT_DIR, exist_ok=True)
print('workspace OK')

Mounted at /content/drive
workspace OK


In [2]:
# Cell 2: Dependencies — torchao MUST go before peft import (runtimes ship
# 0.10; latest peft raises ImportError whenever it is present).
get_ipython().system('pip uninstall -y torchao')
get_ipython().system('pip install -q -U transformers peft accelerate')

# belt-and-braces: verify torchao is really gone before anything imports peft
try:
    import torchao  # noqa: F401
    raise RuntimeError('torchao still importable — restart runtime and Run-all')
except ImportError:
    print('torchao absent: OK')

import torch, gzip, json, time
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.2 MB/s eta 0:00:00
torchao absent: OK
torch 2.11.0+cu130 | cuda True | Tesla T4


In [3]:
# Cell 3: Corpora registry + loader (add a family = add one line)
CORPORA = {
    'MB-85xx': 'MB-85xx_progression_test.jsonl.gz',
    'MB-87xx': 'MB-87xx_liverplasma_test.jsonl.gz',
    'MB-88xx': 'MB-88xx_perturb_test.jsonl.gz',
}

def load_corpus(path):
    recs = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('{"__meta__"'):
                continue
            recs.append(json.loads(line))
    return recs

corpus_records = {}
for tag, fname in CORPORA.items():
    corpus_records[tag] = load_corpus(os.path.join(PROMPTS, fname))
    tasks = sorted({r['task'] for r in corpus_records[tag]})
    print(f'{tag}: {len(corpus_records[tag])} records | tasks {tasks}')

MB-85xx: 104 records | tasks ['MB-8501', 'MB-8505']
MB-87xx: 422 records | tasks ['MB-8701', 'MB-8705']
MB-88xx: 30 records | tasks ['MB-8801']


In [4]:
# Cell 4: Load Qwen3-1.7B base + v5 LoRA adapter FROM DRIVE (unmerged)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def probe_adapter(d):
    """Force Drive FUSE to materialize an adapter dir; fail loudly if empty."""
    files = sorted(os.listdir(d))
    if 'adapter_config.json' not in files:
        raise FileNotFoundError(
            f'adapter_config.json not materialized in {d} ({len(files)} files seen); '
            'remount Drive: drive.flush_and_unmount(); '
            'drive.mount("/content/drive", force_remount=True), then re-run this cell')
    return d

BASE = 'Qwen/Qwen3-1.7B'
ADAPTER = probe_adapter(os.path.join(DRIVE_MASLD, 'checkpoints', 'qwen3-1.7b-masld-sft-v5'))

tok = AutoTokenizer.from_pretrained(BASE, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16)
model = PeftModel.from_pretrained(model, ADAPTER)
model.to('cuda').eval()
print('model ready |', os.path.basename(ADAPTER))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model ready | qwen3-1.7b-masld-sft-v5


In [5]:
# Cell 5: Eval harness — platform local_runner protocol
MAX_NEW_TOKENS = 24
MAX_LENGTH = 4096
MODEL_VERSION = 'qwen3-1.7b-masld-v5'

def evaluate(records, batch=8):
    out, i, t0 = [], 0, time.time()
    while i < len(records):
        chunk = records[i:i + batch]
        try:
            enc = tok(
                [r['prompt'] for r in chunk],
                padding=True, truncation=True, max_length=MAX_LENGTH,
                return_tensors='pt',
            ).to('cuda')
            with torch.no_grad():
                gen = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tok.pad_token_id,
                )
            for j, r in enumerate(chunk):
                text = tok.decode(gen[j][enc['input_ids'].shape[1]:], skip_special_tokens=True)
                out.append({
                    'task': r['task'], 'sample': r['sample'],
                    'answer': r.get('answer'),
                    'raw': text.split('\n')[0].strip(),
                    'error': None, 'model_version': MODEL_VERSION,
                    'parsed': None,
                })
            i += len(chunk)
            if (i // batch) % 10 == 0 or i == len(records):
                rate = i / max(time.time() - t0, 1e-9)
                print(f'  {i}/{len(records)} ({rate:.1f} items/s)', flush=True)
        except RuntimeError as exc:
            if batch > 1:
                batch = max(batch // 2, 1)
                print(f'  batch failure -> halving to {batch}', flush=True)
                continue
            raise
    return out

def write_archive(records, out_path):
    # __meta__ needs n_records + seed + built + builder or the scorer skips it
    with gzip.open(out_path, 'wt', encoding='utf-8') as f:
        f.write(json.dumps({'__meta__': {
            'n_records': len(records), 'seed': 20260929,
            'built': time.strftime('%Y-%m-%d'),
            'builder': 'colab/06_c3_evaluations.ipynb (platform local_runner protocol)',
            'model_version': MODEL_VERSION,
            'protocol': 'greedy, max_new_tokens=24, newline-stopped',
        }}) + '\n')
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print('wrote', out_path)

print('eval harness ready')

eval harness ready


In [6]:
# Cell 6: Evaluate ALL corpora (loop) — rerun safe; overwrites per family
for tag in CORPORA:
    print(f'\n=== {tag} ===')
    recs = evaluate(corpus_records[tag], batch=8)
    write_archive(recs, os.path.join(OUT_DIR, f'{MODEL_VERSION}__{tag}.jsonl.gz'))


=== MB-85xx ===
  80/104 (1.1 items/s)
  104/104 (1.1 items/s)
wrote /content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld-v5__MB-85xx.jsonl.gz

=== MB-87xx ===
  80/422 (3.9 items/s)
  160/422 (3.7 items/s)
  240/422 (3.6 items/s)
  320/422 (3.5 items/s)
  400/422 (3.4 items/s)
  422/422 (3.4 items/s)
wrote /content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld-v5__MB-87xx.jsonl.gz

=== MB-88xx ===
  30/30 (1.2 items/s)
wrote /content/drive/MyDrive/masld-llm/results/frontier_runs/qwen3-1.7b-masld-v5__MB-88xx.jsonl.gz


In [7]:
# Cell 7: Verify + peek (exact-match vs majority; scorer re-parses raw)
from collections import Counter

for tag in CORPORA:
    path = os.path.join(OUT_DIR, f'{MODEL_VERSION}__{tag}.jsonl.gz')
    recs = load_corpus(path)
    print(f'\n=== {tag}: {len(recs)} records ===')
    for task in sorted({r["task"] for r in recs}):
        sub = [r for r in recs if r['task'] == task]
        match = sum(1 for r in sub if str(r['raw']).strip().lower() == str(r['answer']).strip().lower())
        maj = Counter(str(r['answer']) for r in sub).most_common(1)[0]
        print(f'{task}: n={len(sub)} | exact-match {match/len(sub):.3f} '
              f'| majority({maj[0]!r})={maj[1]/len(sub):.3f}')
        print('  raw top3:', Counter(r['raw'] for r in sub).most_common(3))


=== MB-85xx: 104 records ===
MB-8501: n=52 | exact-match 0.173 | majority('stable')=0.904
  raw top3: [('PROGRESSIVE', 32), ('progressive', 11), ('STABLE', 4)]
MB-8505: n=52 | exact-match 0.635 | majority('1')=0.904
  raw top3: [('1', 36), ('1 or 2 2', 8), ('1 or 2', 5)]

=== MB-87xx: 422 records ===
MB-8701: n=206 | exact-match 0.257 | majority('Plasma_liver2')=0.350
  raw top3: [('Plasma_liver2', 111), ('Lumos72', 95)]
MB-8705: n=216 | exact-match 0.000 | majority('-0.31')=0.019
  raw top3: [('-1.00', 101), ('1.00', 6), ('-1.05', 4)]

=== MB-88xx: 30 records ===
MB-8801: n=30 | exact-match 0.433 | majority('no')=0.500
  raw top3: [('no', 28), ('yes', 2)]


In [8]:
# Cell 8 (optional): swap to 0.6B-v5 and evaluate the same corpora.
# Hardened: torchao re-check (runtimes can restore it) + adapter FUSE probe.
RUN_06B = False  # flip to True to run

if RUN_06B:
    get_ipython().system('pip uninstall -y torchao')
    import gc
    ADAPTER06 = probe_adapter(
        os.path.join(DRIVE_MASLD, 'checkpoints', 'qwen3-0.6b-masld-sft-v5'))
    del model
    gc.collect(); torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B', dtype=torch.float16)
    model = PeftModel.from_pretrained(model, ADAPTER06)
    model.to('cuda').eval()
    MODEL_VERSION = 'qwen3-0.6b-masld-v5'
    globals()['MODEL_VERSION'] = MODEL_VERSION
    for tag in CORPORA:
        print(f'\n=== {tag} (0.6B-v5) ===')
        recs = evaluate(corpus_records[tag], batch=8)
        write_archive(recs, os.path.join(OUT_DIR, f'{MODEL_VERSION}__{tag}.jsonl.gz'))
else:
    print('skipped (RUN_06B = False)')

skipped (RUN_06B = False)
